# TicTacToe × Reinforcement Learning
## Encodings — Description de l'état et des actions

> **Référence** : Sutton & Barto, *Reinforcement Learning: An Introduction*, Chap. 3 — *The Reinforcement Learning Problem*

---

Ce notebook décrit et illustre :
1. **L'espace d'états** — comment représenter le plateau sous forme de vecteur numérique
2. **L'espace d'actions** — comment encoder un coup joué
3. **Les propriétés** de ces espaces (taille, légalité, symétries)
4. **Des exemples concrets** avec visualisations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import itertools

# Palette cohérente avec la GUI
COLORS = {
    'bg':     '#0e0f14',
    'panel':  '#1c1e28',
    'border': '#2a2d3e',
    'accent': '#e8c547',
    'x':      '#e05c5c',
    'o':      '#5ca8e0',
    'text':   '#d8daea',
    'muted':  '#6b6f8a',
    'green':  '#5ce07a',
}

plt.rcParams.update({
    'figure.facecolor':  COLORS['bg'],
    'axes.facecolor':    COLORS['panel'],
    'axes.edgecolor':    COLORS['border'],
    'axes.labelcolor':   COLORS['text'],
    'text.color':        COLORS['text'],
    'xtick.color':       COLORS['muted'],
    'ytick.color':       COLORS['muted'],
    'grid.color':        COLORS['border'],
    'font.family':       'monospace',
})

print('✓ Imports OK')

---
## 1. Description du plateau

Le plateau TicTacToe est une grille **3 × 3**. On l'aplatit en un **vecteur de longueur 9**.

```
Indexation des cases :

  0 | 1 | 2
  ---------
  3 | 4 | 5
  ---------
  6 | 7 | 8

  case (row, col)  →  index = row × 3 + col
```

In [ ]:
def draw_index_grid():
    """Visualise l'indexation des cases du plateau."""
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_aspect('equal')
    ax.axis('off')
    fig.patch.set_facecolor(COLORS['bg'])
    ax.set_facecolor(COLORS['panel'])

    # Grille
    for i in range(1, 3):
        ax.axhline(i, color=COLORS['border'], lw=2)
        ax.axvline(i, color=COLORS['border'], lw=2)

    # Index
    for i in range(9):
        r, c = divmod(i, 3)
        ax.text(c + 0.5, 2.5 - r, str(i),
                ha='center', va='center',
                fontsize=22, fontweight='bold',
                color=COLORS['accent'])
        ax.text(c + 0.5, 2.18 - r, f'({r},{c})',
                ha='center', va='center',
                fontsize=9, color=COLORS['muted'])

    ax.set_title('Indexation des cases\nindex = row × 3 + col',
                 color=COLORS['text'], pad=12, fontsize=11)
    plt.tight_layout()
    plt.show()

draw_index_grid()

---
## 2. Encodage de l'état — State Encoding

### 2.1 Définition formelle

| Valeur | Signification |
|--------|---------------|
| `0`    | Case vide |
| `+1`   | Joueur 1 — Agent RL (symbole **X**) |
| `-1`   | Joueur 2 — Adversaire random (symbole **O**) |

$$s \in \{-1,\ 0,\ +1\}^9 \quad \text{(vecteur float32 de longueur 9)}$$

### 2.2 Propriétés

| Propriété | Valeur |
|-----------|--------|
| Taille du vecteur | 9 |
| Type numpy | `float32` |
| Valeurs par dimension | `{-1.0, 0.0, 1.0}` |
| Nombre d'états théoriques | $3^9 = 19\,683$ |
| Nombre d'états légaux | $\approx 5\,478$ |
| Symétries exploitables | 8 (4 rotations × 2 réflexions) |

> **Note** : pour le `TabularQLearning`, l'état est converti en clé de dictionnaire via `tuple(state.astype(int))`. La Q-table a alors au plus **5 478 × 9** entrées.

In [ ]:
def encode_state(board_2d: np.ndarray) -> np.ndarray:
    """
    Encode un plateau 3×3 en vecteur d'état plat.

    Paramètres
    ----------
    board_2d : np.ndarray (3, 3) — valeurs {-1, 0, 1}

    Retourne
    --------
    state : np.ndarray (9,) float32
    """
    return board_2d.flatten().astype(np.float32)


def state_to_qtable_key(state: np.ndarray) -> tuple:
    """
    Convertit le vecteur d'état en clé hashable pour une Q-table.
    Utilisé par TabularQLearning.
    """
    return tuple(state.astype(int))


# ── Exemple ──────────────────────────────────────────────────────
board_example = np.array([
    [ 1,  0, -1],
    [ 0,  1,  0],
    [ 0,  0, -1]
], dtype=np.int8)

state = encode_state(board_example)
key   = state_to_qtable_key(state)

print('Plateau 2D :')
print(board_example)
print()
print(f'Vecteur d\'état (9,) float32 :')
print(f'  state = {state}')
print()
print(f'Clé Q-table :')
print(f'  key   = {key}')

In [ ]:
def visualize_state(board_flat: np.ndarray, title: str = 'État du jeu'):
    """
    Affiche côte à côte :
      - le plateau de jeu
      - le vecteur d'état correspondant
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                             gridspec_kw={'width_ratios': [1, 1.6]})
    fig.patch.set_facecolor(COLORS['bg'])
    fig.suptitle(title, color=COLORS['accent'], fontsize=13, y=1.02)

    # ── Plateau ──────────────────────────────────────────────────
    ax = axes[0]
    ax.set_xlim(0, 3); ax.set_ylim(0, 3)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_facecolor(COLORS['panel'])

    for i in range(1, 3):
        ax.axhline(i, color=COLORS['border'], lw=2)
        ax.axvline(i, color=COLORS['border'], lw=2)

    sym  = {1: 'X', -1: 'O', 0: '·'}
    col  = {1: COLORS['x'], -1: COLORS['o'], 0: COLORS['border']}

    for i, v in enumerate(board_flat):
        r, c = divmod(i, 3)
        ax.text(c + 0.5, 2.5 - r, sym[int(v)],
                ha='center', va='center',
                fontsize=28, fontweight='bold',
                color=col[int(v)])
        ax.text(c + 0.12, 2.12 - r, str(i),
                ha='center', va='center',
                fontsize=7, color=COLORS['muted'])

    ax.set_title('Plateau', color=COLORS['text'], fontsize=10)

    # ── Vecteur d'état ───────────────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor(COLORS['bg'])
    ax2.axis('off')

    cell_w, cell_h = 0.09, 0.28
    x_start = 0.02

    bg_col  = {1: COLORS['x'],  -1: COLORS['o'],  0: COLORS['border']}
    txt_col = {1: '#ffffff',    -1: '#ffffff',     0: COLORS['muted']}
    val_lbl = {1: '+1',         -1: '−1',          0: '0'}

    # Titre
    ax2.text(0.5, 0.92, 'Vecteur d\'état  s ∈ {−1, 0, +1}⁹',
             ha='center', va='center', fontsize=10,
             color=COLORS['text'],
             transform=ax2.transAxes)

    for i, v in enumerate(board_flat):
        x = x_start + i * (cell_w + 0.01)
        v_int = int(v)

        # Fond coloré
        rect = mpatches.FancyBboxPatch(
            (x, 0.52), cell_w, cell_h,
            boxstyle='round,pad=0.01',
            facecolor=bg_col[v_int],
            edgecolor='none',
            transform=ax2.transAxes,
            alpha=0.85
        )
        ax2.add_patch(rect)

        # Valeur
        ax2.text(x + cell_w/2, 0.52 + cell_h/2, val_lbl[v_int],
                 ha='center', va='center',
                 fontsize=11, fontweight='bold',
                 color=txt_col[v_int],
                 transform=ax2.transAxes)

        # Index
        ax2.text(x + cell_w/2, 0.46, f'i={i}',
                 ha='center', va='center',
                 fontsize=7, color=COLORS['muted'],
                 transform=ax2.transAxes)

    # Légende
    legend_items = [
        mpatches.Patch(color=COLORS['x'],      label='X = +1  (Agent RL)'),
        mpatches.Patch(color=COLORS['o'],      label='O = −1  (Random)'),
        mpatches.Patch(color=COLORS['border'], label='·  =  0  (Vide)'),
    ]
    ax2.legend(handles=legend_items, loc='lower center',
               fontsize=8, framealpha=0.2,
               facecolor=COLORS['panel'],
               edgecolor=COLORS['border'],
               labelcolor=COLORS['text'])

    plt.tight_layout()
    plt.show()


# ── Exemple 1 : état initial ──────────────────────────────────────
state_init = np.zeros(9, dtype=np.float32)
visualize_state(state_init, 'État initial — plateau vide')

In [ ]:
# ── Exemple 2 : état intermédiaire ───────────────────────────────
#   X | · | O
#   · | X | ·
#   · | · | O
state_mid = np.array([1, 0, -1, 0, 1, 0, 0, 0, -1], dtype=np.float32)
visualize_state(state_mid, 'État intermédiaire')

In [ ]:
# ── Exemple 3 : état terminal (X gagne en diagonale) ─────────────
#   X | O | O
#   · | X | ·
#   O | · | X
state_win = np.array([1, -1, -1, 0, 1, 0, -1, 0, 1], dtype=np.float32)
visualize_state(state_win, 'État terminal — X gagne (diagonale 0→4→8)')

---
## 3. Masque d'actions légales — Action Mask

À chaque état, toutes les actions ne sont pas disponibles. On définit un **masque binaire** :

$$\text{mask}[i] = \begin{cases} 1 & \text{si } s[i] = 0 \text{ (case libre)} \\ 0 & \text{sinon} \end{cases}$$

Ce masque est essentiel pour les réseaux de neurones : on l'applique à la sortie avant `argmax` pour **interdire les actions illégales**.

In [ ]:
def get_action_mask(state: np.ndarray) -> np.ndarray:
    """
    Retourne le masque binaire des actions légales.

    mask[i] = 1.0  →  case i libre (action légale)
    mask[i] = 0.0  →  case i occupée (action illégale)
    """
    return (state == 0).astype(np.float32)


def get_valid_actions(state: np.ndarray) -> list:
    """Liste des indices des actions légales."""
    return [i for i in range(9) if state[i] == 0]


# Exemple sur l'état intermédiaire
mask   = get_action_mask(state_mid)
valids = get_valid_actions(state_mid)

print('État           :', state_mid.astype(int))
print('Masque légal   :', mask.astype(int))
print('Actions légales:', valids)
print()
print('Application dans un DQN :')
print('  q_masked = q_values + (-1e9) * (1 - mask)')
print('  action   = np.argmax(q_masked)')

---
## 4. Encodage d'une action — Action Encoding

### 4.1 Définition formelle

Une **action** = choisir une case libre parmi les 9 disponibles.

**Encodage entier** (utilisé pour Q-Learning, indexation Q-table) :
$$a \in \{0, 1, 2, 3, 4, 5, 6, 7, 8\}$$

**Encodage one-hot** (utilisé pour réseaux de neurones, REINFORCE, PPO) :
$$\mathbf{a} \in \{0, 1\}^9, \quad \mathbf{a}[i] = \begin{cases} 1 & \text{si } i = a \\ 0 & \text{sinon} \end{cases}$$

### 4.2 Propriétés

| Propriété | Valeur |
|-----------|--------|
| Type d'espace | Discret |
| Taille totale | 9 |
| Actions légales | Cases où `state[i] = 0` (variable) |
| Encodage entier | `a ∈ {0…8}` |
| Encodage one-hot | `vecteur ∈ {0,1}⁹` |
| Conversion `(row, col)` → `a` | `a = row × 3 + col` |
| Conversion `a` → `(row, col)` | `row, col = divmod(a, 3)` |

In [ ]:
def encode_action_onehot(action: int) -> np.ndarray:
    """
    Encode l'action en vecteur one-hot de longueur 9.

    Paramètre
    ---------
    action : int ∈ {0…8}

    Retourne
    --------
    one_hot : np.ndarray (9,) float32
    """
    one_hot = np.zeros(9, dtype=np.float32)
    one_hot[action] = 1.0
    return one_hot


def action_to_cell(action: int) -> tuple:
    """Convertit un index d'action en coordonnées (row, col)."""
    return divmod(action, 3)


def cell_to_action(row: int, col: int) -> int:
    """Convertit des coordonnées (row, col) en index d'action."""
    return row * 3 + col


# ── Affichage de tous les encodages one-hot ───────────────────────
print('Action  (row,col)  One-hot encoding')
print('─' * 50)
for a in range(9):
    r, c = action_to_cell(a)
    oh   = encode_action_onehot(a)
    print(f'  a={a}   ({r},{c})       {oh.astype(int)}')

In [ ]:
def visualize_action(action: int, state: np.ndarray = None):
    """
    Affiche côte à côte :
      - le plateau avec l'action mise en évidence
      - le vecteur one-hot correspondant
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                             gridspec_kw={'width_ratios': [1, 1.6]})
    fig.patch.set_facecolor(COLORS['bg'])
    r_a, c_a = action_to_cell(action)
    fig.suptitle(f'Action a = {action}  →  case ({r_a}, {c_a})',
                 color=COLORS['accent'], fontsize=13, y=1.02)

    board = state if state is not None else np.zeros(9)

    # ── Plateau ──────────────────────────────────────────────────
    ax = axes[0]
    ax.set_xlim(0, 3); ax.set_ylim(0, 3)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_facecolor(COLORS['panel'])

    for i in range(1, 3):
        ax.axhline(i, color=COLORS['border'], lw=2)
        ax.axvline(i, color=COLORS['border'], lw=2)

    sym = {1: 'X', -1: 'O', 0: ''}
    col = {1: COLORS['x'], -1: COLORS['o'], 0: COLORS['muted']}

    for i in range(9):
        r, c = divmod(i, 3)
        v = int(board[i])

        if i == action:
            rect = plt.Rectangle((c, 2 - r), 1, 1,
                                  facecolor=COLORS['accent'],
                                  alpha=0.25, zorder=1)
            ax.add_patch(rect)
            ax.text(c + 0.5, 2.5 - r, '★',
                    ha='center', va='center',
                    fontsize=26, color=COLORS['accent'], zorder=2)
        else:
            if sym[v]:
                ax.text(c + 0.5, 2.5 - r, sym[v],
                        ha='center', va='center',
                        fontsize=26, fontweight='bold',
                        color=col[v], zorder=2)

        ax.text(c + 0.12, 2.12 - r, str(i),
                ha='center', va='center',
                fontsize=7, color=COLORS['muted'])

    ax.set_title(f'Case choisie : index {action}',
                 color=COLORS['text'], fontsize=10)

    # ── Vecteur one-hot ──────────────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor(COLORS['bg'])
    ax2.axis('off')

    one_hot = encode_action_onehot(action)
    cell_w, cell_h = 0.09, 0.28
    x_start = 0.02

    ax2.text(0.5, 0.92, 'Encodage one-hot  a ∈ {0,1}⁹',
             ha='center', va='center', fontsize=10,
             color=COLORS['text'], transform=ax2.transAxes)

    for i, v in enumerate(one_hot):
        x    = x_start + i * (cell_w + 0.01)
        is_a = (i == action)
        fc   = COLORS['accent'] if is_a else COLORS['border']
        tc   = COLORS['bg']     if is_a else COLORS['muted']

        rect = mpatches.FancyBboxPatch(
            (x, 0.52), cell_w, cell_h,
            boxstyle='round,pad=0.01',
            facecolor=fc, edgecolor='none',
            transform=ax2.transAxes,
            alpha=0.9
        )
        ax2.add_patch(rect)

        ax2.text(x + cell_w/2, 0.52 + cell_h/2,
                 str(int(v)),
                 ha='center', va='center',
                 fontsize=13, fontweight='bold',
                 color=tc, transform=ax2.transAxes)

        ax2.text(x + cell_w/2, 0.46, f'i={i}',
                 ha='center', va='center',
                 fontsize=7, color=COLORS['muted'],
                 transform=ax2.transAxes)

    ax2.text(0.5, 0.28,
             f'one_hot[{action}] = 1  (tous les autres = 0)',
             ha='center', va='center', fontsize=9,
             color=COLORS['accent'], transform=ax2.transAxes)

    plt.tight_layout()
    plt.show()


# Exemples
visualize_action(4)   # centre
visualize_action(0)   # coin haut-gauche
visualize_action(7, state=state_mid)  # action sur état intermédiaire

---
## 5. Récapitulatif complet et guide d'utilisation

### 5.1 Tableau récapitulatif

| | **État** | **Action** |
|---|---|---|
| **Taille** | 9 | 9 |
| **Type** | `float32` | `int` ou `float32` (one-hot) |
| **Valeurs** | `{−1.0, 0.0, +1.0}` | `{0…8}` ou `{0,1}⁹` |
| **Q-table** | `tuple(s.astype(int))` | index direct |
| **Réseau** | vecteur d'entrée (9 neurones) | 9 logits en sortie |
| **Masque légal** | — | `mask = (state == 0)` |

### 5.2 Utilisation par algorithme

| Algorithme | Encodage état | Encodage action |
|---|---|---|
| **TabularQLearning** | `tuple(s.astype(int))` | entier `a ∈ {0…8}` |
| **DQN / DDQN** | vecteur float32 (9) | argmax sur 9 logits masqués |
| **REINFORCE / PPO** | vecteur float32 (9) | softmax masquée → `log π(a∣s)` |
| **MCTS / AlphaZero** | vecteur float32 (9) | distribution sur 9 cases |
| **MuZero** | vecteur float32 (9) | distribution sur 9 cases |

In [ ]:
# ── Visualisation de l'espace d'états légaux ─────────────────────

def count_legal_states():
    """
    Compte les états légaux du TicTacToe par force brute.
    Un état est légal si :
      - |nb_X - nb_O| <= 1  (X commence)
      - pas deux gagnants simultanés
    """
    WINNING = [[0,1,2],[3,4,5],[6,7,8],
               [0,3,6],[1,4,7],[2,5,8],
               [0,4,8],[2,4,6]]

    def wins(b, p):
        return any(all(b[i] == p for i in l) for l in WINNING)

    legal = 0
    for combo in itertools.product([-1, 0, 1], repeat=9):
        b  = list(combo)
        nx = b.count(1)
        no = b.count(-1)
        if not (no <= nx <= no + 1):
            continue
        wx, wo = wins(b, 1), wins(b, -1)
        if wx and wo:
            continue
        if wx and nx != no + 1:
            continue
        if wo and nx != no:
            continue
        legal += 1
    return legal

n_legal = count_legal_states()

print('━' * 50)
print('  ESPACE D\'ÉTATS — TicTacToe')
print('━' * 50)
print(f'  États théoriques (3^9)  : {3**9:>8,}')
print(f'  États légaux            : {n_legal:>8,}')
print(f'  Ratio légaux/théoriques : {n_legal/3**9*100:>7.1f} %')
print()
print(f'  Q-table (TabularQL)     : {n_legal} × 9 = {n_legal*9:,} entrées')
print('━' * 50)

In [ ]:
# ── Figure de synthèse finale ─────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.patch.set_facecolor(COLORS['bg'])
fig.suptitle('TicTacToe × RL — Récapitulatif des Encodings',
             color=COLORS['accent'], fontsize=14, y=1.03)

# ── Schéma 1 : état initial ───────────────────────────────────────
ax = axes[0]
ax.set_facecolor(COLORS['panel'])
ax.set_xlim(0, 3); ax.set_ylim(0, 3)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('État initial\ns = [0,0,0,0,0,0,0,0,0]',
             color=COLORS['text'], fontsize=9)
for i in range(1, 3):
    ax.axhline(i, color=COLORS['border'], lw=1.5)
    ax.axvline(i, color=COLORS['border'], lw=1.5)
for i in range(9):
    r, c = divmod(i, 3)
    ax.text(c+0.5, 2.5-r, str(i), ha='center', va='center',
            fontsize=16, color=COLORS['muted'])

# ── Schéma 2 : état intermédiaire ────────────────────────────────
ax = axes[1]
ax.set_facecolor(COLORS['panel'])
ax.set_xlim(0, 3); ax.set_ylim(0, 3)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('État intermédiaire\ns = [1,0,-1,0,1,0,0,0,-1]',
             color=COLORS['text'], fontsize=9)
for i in range(1, 3):
    ax.axhline(i, color=COLORS['border'], lw=1.5)
    ax.axvline(i, color=COLORS['border'], lw=1.5)
board = [1, 0, -1, 0, 1, 0, 0, 0, -1]
sym2  = {1: 'X', -1: 'O', 0: '·'}
col2  = {1: COLORS['x'], -1: COLORS['o'], 0: COLORS['border']}
for i, v in enumerate(board):
    r, c = divmod(i, 3)
    ax.text(c+0.5, 2.5-r, sym2[v], ha='center', va='center',
            fontsize=24, fontweight='bold', color=col2[v])

# ── Schéma 3 : action one-hot ────────────────────────────────────
ax = axes[2]
ax.set_facecolor(COLORS['bg'])
ax.axis('off')
ax.set_title('Action a=4 (centre)\none-hot = [0,0,0,0,1,0,0,0,0]',
             color=COLORS['text'], fontsize=9)
oh = encode_action_onehot(4)
colors_bar = [COLORS['accent'] if i == 4 else COLORS['border'] for i in range(9)]
bars = ax.bar(range(9), oh, color=colors_bar, width=0.7, zorder=2)
ax.set_xticks(range(9))
ax.set_xticklabels([f'a={i}' for i in range(9)], fontsize=7,
                   color=COLORS['muted'])
ax.set_yticks([0, 1])
ax.set_ylim(-0.1, 1.3)
ax.tick_params(colors=COLORS['muted'])
ax.spines[:].set_color(COLORS['border'])
ax.text(4, 1.1, '1', ha='center', fontsize=12,
        fontweight='bold', color=COLORS['accent'])
ax.set_facecolor(COLORS['panel'])

plt.tight_layout()
plt.show()

print('\n✓ Notebook complet — tous les encodings sont définis et illustrés.')